# 04 Recurrent neural networks: BiLSTM and BiGRU with attention

**Questions**
1. Does a sequence model (which reads words in order) beat bag-of-words on this small dataset?
2. Do Word2Vec embeddings trained on our articles help compared with random embeddings? Should they be frozen or fine-tuned?
3. LSTM or GRU? Attention pooling or the last hidden state?
4. Do class weights help, as they did for logistic regression?

**Architecture** (see `src/rnn_models.py`):
`words → embedding (100-d) → bidirectional LSTM/GRU (2 × 128) → attention pooling → dropout → linear → softmax over 4 topics`.

Training: Adam (lr 1e-3), batch size 16, cross-entropy loss (optionally class-weighted), gradient clipping at 1.0, up to
25 epochs with **early stopping** when validation macro-F1 has not improved for 6 epochs; we keep the best epoch's weights.
Input is capped at 300 words (95% of articles fit whole).

Each experiment changes **one thing** relative to the reference run R2. Exploration uses one seed (42); the final configuration
is retrained with 3 seeds and we report the mean and standard deviation.

In [ ]:
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

# Change this to your own GitHub repository before running in Colab
REPO_URL = "https://github.com/YOUR-GITHUB-USERNAME/lingala-news-classification.git"

if IN_COLAB and not Path("lingala-news-classification").exists():
    !git clone -q {REPO_URL}
if IN_COLAB:
    %cd lingala-news-classification
    %pip install -q -r requirements.txt

if Path.cwd().name == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
print("Working directory:", os.getcwd())

In [ ]:
import time

import pandas as pd
import torch
from gensim.models import Word2Vec

from src import config, data_loading, evaluation, plots, rnn_models
from src.experiment_log import log_experiment
from src.word_embeddings import embedding_matrix_for_vocabulary

MODEL_FAMILY = "rnn"
torch.set_num_threads(max(1, torch.get_num_threads()))

train, validation, test = data_loading.load_all_splits()
validation_ids = data_loading.labels_to_ids(validation["category"])
test_ids = data_loading.labels_to_ids(test["category"])

word2vec = Word2Vec.load(str(config.MODELS_DIRECTORY / "word2vec_lingala_100d.model"))

def word2vec_embeddings(vocabulary, seed=config.RANDOM_SEED):
    matrix, coverage = embedding_matrix_for_vocabulary(vocabulary, word2vec, seed)
    print(f"  Word2Vec covers {coverage:.0%} of the RNN vocabulary")
    return matrix

In [ ]:
REFERENCE = {
    "cell": "lstm", "pooling": "attention", "embeddings": "word2vec", "freeze_embeddings": False,
    "embedding_dimensions": 100, "hidden_size": 128, "dropout": 0.3, "embedding_dropout": 0.2,
    "learning_rate": 1e-3, "batch_size": 16, "epochs": 25, "patience": 6,
    "min_count": 2, "max_length": 300, "class_weight": True,
}

results, histories = [], {}

def run(name, changes, notes, seed=config.RANDOM_SEED):
    settings = {**REFERENCE, **changes}
    started = time.time()
    embeddings_fn = word2vec_embeddings if settings["embeddings"] == "word2vec" else None
    model, vocabulary, history = rnn_models.train_recurrent_model(settings, train, validation, seed, embeddings_fn)
    probabilities = rnn_models.predict_dataframe(model, vocabulary, validation, settings)
    scores = evaluation.compute_scores(validation_ids, probabilities)
    scores["best_epoch"] = history["best_epoch"]
    log_experiment(MODEL_FAMILY, name, "validation", {**settings, "seed": seed}, scores, notes)
    results.append({"experiment": name, "change": notes, **{k: scores[k] for k in ["macro_f1", "weighted_f1", "log_loss"]},
                    "best_epoch": history["best_epoch"], "minutes": round((time.time() - started) / 60, 1)})
    histories[name] = history
    print(f"{name}: validation macro-F1 {scores['macro_f1']:.3f} (best epoch {history['best_epoch']}, {results[-1]['minutes']} min)")
    return model, vocabulary, settings

In [ ]:
run("R1_bilstm_random_embeddings", {"embeddings": "random"}, "Random embeddings instead of Word2Vec")
run("R2_bilstm_word2vec_finetuned", {}, "Reference: Word2Vec init, fine-tuned")
run("R3_bilstm_word2vec_frozen", {"freeze_embeddings": True}, "Word2Vec frozen (not updated)")
run("R4_bilstm_last_state", {"pooling": "last"}, "Last hidden state instead of attention")
run("R5_bigru_attention", {"cell": "gru"}, "GRU instead of LSTM")
run("R6_bilstm_no_class_weights", {"class_weight": False}, "Unweighted cross-entropy");

In [ ]:
summary = pd.DataFrame(results)
summary

In [ ]:
for name in ["R1_bilstm_random_embeddings", "R2_bilstm_word2vec_finetuned"]:
    plots.plot_learning_curves(histories[name], "rnn", name)

**How to read the learning curves.** Training loss keeps falling while validation loss flattens or rises after a few
epochs: the network starts memorising the 608 training articles. Early stopping keeps the epoch with the best validation
macro-F1. The validation macro-F1 curve is jumpy because one article out of 8 business or 10 sports articles moves it by
several points; this is why the final model is run with three seeds.

## Final configuration: three seeds, evaluated on the test set

In [ ]:
best = summary.sort_values(["macro_f1", "log_loss"], ascending=[False, True]).iloc[0]["experiment"]
changes = {
    "R1_bilstm_random_embeddings": {"embeddings": "random"},
    "R2_bilstm_word2vec_finetuned": {},
    "R3_bilstm_word2vec_frozen": {"freeze_embeddings": True},
    "R4_bilstm_last_state": {"pooling": "last"},
    "R5_bigru_attention": {"cell": "gru"},
    "R6_bilstm_no_class_weights": {"class_weight": False},
}[best]
print("Final configuration:", best, changes)

seed_scores, seed_models = [], {}
for seed in config.FINAL_RUN_SEEDS:
    settings = {**REFERENCE, **changes}
    embeddings_fn = (lambda v, s=seed: word2vec_embeddings(v, s)) if settings["embeddings"] == "word2vec" else None
    model, vocabulary, history = rnn_models.train_recurrent_model(settings, train, validation, seed, embeddings_fn)
    probabilities = rnn_models.predict_dataframe(model, vocabulary, test, settings)
    scores = evaluation.compute_scores(test_ids, probabilities)
    seed_scores.append(scores)
    seed_models[seed] = (model, vocabulary, probabilities)
    plots.plot_learning_curves(history, "rnn", f"final_seed_{seed}")
    print(f"seed {seed}: test macro-F1 {scores['macro_f1']:.3f}")

summary_scores = evaluation.summarise_runs(seed_scores)
first_seed = config.FINAL_RUN_SEEDS[0]
low, high = evaluation.bootstrap_confidence_interval(test_ids, seed_models[first_seed][2].argmax(1))
summary_scores["macro_f1_ci_low"], summary_scores["macro_f1_ci_high"] = low, high
log_experiment(MODEL_FAMILY, f"FINAL_{best}", "test", {**REFERENCE, **changes, "seeds": config.FINAL_RUN_SEEDS},
               summary_scores, "Mean and std over 3 seeds; CI from seed 42")
evaluation.save_test_predictions(seed_models[first_seed][2], "rnn")
print(f"Test macro-F1 {summary_scores['macro_f1']:.3f} ± {summary_scores['macro_f1_std']:.3f} over {len(seed_scores)} seeds")
evaluation.print_classification_report(test_ids, seed_models[first_seed][2])
plots.plot_confusion_matrix(test_ids, seed_models[first_seed][2], "rnn", "test");

## What does the attention look at?
We use the attention model (the final model if it uses attention, otherwise the reference run R2). For a few test articles we list the words with the highest attention weight. If attention is useful, the top words should be
topic words, not function words. (Attention weights are a hint about what the model used, not a full explanation; Jain & Wallace, 2019.)

In [ ]:
final_settings = {**REFERENCE, **changes}
if final_settings["pooling"] == "attention":
    model, vocabulary, probabilities = seed_models[first_seed]
    attention_settings = final_settings
else:
    # The final model uses the last hidden state, so we inspect the attention model R2 instead
    attention_settings = REFERENCE
    model, vocabulary, _ = rnn_models.train_recurrent_model(REFERENCE, train, validation, first_seed, word2vec_embeddings)
    probabilities = rnn_models.predict_dataframe(model, vocabulary, test, REFERENCE)
if True:
    table = evaluation.build_prediction_table(test, test_ids, probabilities)
    examples = pd.concat([table[table["correct"]].groupby("true_label").head(1),
                          table[~table["correct"]].sort_values("confidence", ascending=False).head(4)])
    rows = []
    for index, row in examples.iterrows():
        top_words = rnn_models.attention_for_text(model, vocabulary, row["content"], attention_settings, top=8)
        rows.append({"headline": row["headline"][:70], "true": row["true_label"], "predicted": row["predicted_label"],
                     "top attention words": ", ".join(w for w, _ in top_words)})
    attention_table = pd.DataFrame(rows)
    attention_table.to_csv(config.RESULTS_DIRECTORY / "figures" / "rnn" / "attention_examples.csv", index=False)
    display(attention_table)